# dive - Colab quickstart

Train a tabular model, validate the data, and read an HTML report - all from
shell commands, with no local setup.

Runtime: about two minutes end to end.


## 1. Install

Clone the repo and install it. `[full]` pulls in XGBoost, LightGBM, CatBoost,
Optuna, SHAP and category-encoders; the tool works without them, just with a
smaller model zoo.


In [ ]:
!git clone -q https://github.com/Aman-i1/DIVE.git
%cd DIVE
!pip install -q -e ".[full]"


Confirm the `dive` command is on PATH and see which extras are active:


In [ ]:
!dive --version
!dive deps


## 2. Check the data before training

`validate` trains nothing. It looks for the faults that quietly invalidate a
model: a leaking column, duplicate rows spanning the train/holdout split,
distribution drift, a near-constant target.

Run this first - it is much cheaper than discovering the problem after an
hour of training.


In [ ]:
!dive validate --data examples/sample.csv --target diagnosis


### What a caught leak looks like

Here we deliberately add a column that encodes the answer. This is the single
most common way a model scores 99% in testing and fails in production.


In [ ]:
import pandas as pd

frame = pd.read_csv('examples/sample.csv')
frame['leaked_outcome'] = (frame['diagnosis'] == 'M').astype(int)
frame.to_csv('/content/leaky.csv', index=False)

!dive validate --data /content/leaky.csv --target diagnosis


The check fails and names the offending column. Exit code 1, so this works as
a CI gate too.


## 3. Train

`fast` finishes in seconds - a small zoo, no tuning, no stacking. Use
`balanced` for real work, or `competition` for maximum accuracy.

Every mode respects `--time-budget` (seconds) and reports progress as it goes.


In [ ]:
!dive train --data examples/sample.csv --target diagnosis --mode fast --time-budget 300 --output /content/out


For the full zoo with tuning and stacking, swap the mode. On this dataset it
takes roughly 30 seconds:


In [ ]:
!dive train --data examples/sample.csv --target diagnosis --mode balanced --time-budget 600 --output /content/out_balanced


## 4. Read the report inline

The report is one self-contained HTML file - plots are embedded as base64, so
it renders directly in the notebook and can be downloaded on its own.


In [ ]:
from IPython.display import HTML, display

display(HTML(open('/content/out_balanced/report.html', encoding='utf-8').read()))


## 5. Inspect the leaderboard


In [ ]:
import pandas as pd

pd.read_csv('/content/out_balanced/leaderboard.csv')


## 6. Understand what it did

`explain` describes every pipeline stage in plain English, and with `--output`
also emits standalone Python that rebuilds the model without dive.


In [ ]:
!dive explain --model /content/out_balanced/model.pkl


In [ ]:
!dive explain --model /content/out_balanced/model.pkl --output /content/explanation.html

from IPython.display import HTML, display

display(HTML(open('/content/explanation.html', encoding='utf-8').read()))


## 7. Score new rows

The incoming schema is checked against training first. A missing feature
column stops the run rather than silently misaligning columns.


In [ ]:
import pandas as pd

new_rows = pd.read_csv('examples/sample.csv').drop(columns=['diagnosis']).head(15)
new_rows.to_csv('/content/new_rows.csv', index=False)

!dive predict --model /content/out_balanced/model.pkl --data /content/new_rows.csv --output /content/predictions.csv --proba


In [ ]:
pd.read_csv('/content/predictions.csv').head(10)


### Schema enforcement in action

Dropping a required column produces a clear error, not a traceback:


In [ ]:
broken = pd.read_csv('/content/new_rows.csv').drop(columns=['mean_radius'])
broken.to_csv('/content/broken.csv', index=False)

!dive predict --model /content/out_balanced/model.pkl --data /content/broken.csv --output /content/x.csv


## 8. Use your own data

Upload a CSV and point the same commands at it. The only thing that changes is
`--data` and `--target`.


In [ ]:
# from google.colab import files
# uploaded = files.upload()

# !dive validate --data your_file.csv --target your_target_column
# !dive train --data your_file.csv --target your_target_column --mode balanced --output /content/my_run


## 9. Use it as a Python library

The CLI is a wrapper around a normal Python API.


In [ ]:
import pandas as pd
from dive import Dive

frame = pd.read_csv('examples/sample.csv')

model = Dive(target='diagnosis', mode='fast', time_budget=180)
model.fit(frame)
print(model.leaderboard())


In [ ]:
model.feature_importances(top_n=10)


## 10. Download the results


In [ ]:
# from google.colab import files

# files.download('/content/out_balanced/report.html')
# files.download('/content/out_balanced/model.pkl')


---

### Where the output files land

| File | Contents |
|---|---|
| `model.pkl` | Fitted model + feature engineer, used by `dive predict` |
| `leaderboard.csv` | Every model and its scores, best first |
| `report.html` | Self-contained report with plots inlined |
| `validation.json` | Machine-readable crosscheck verdicts |
| `metadata.json` | Run settings, schema, timings |
| `plots/*.png` | Individual diagnostic figures |

Full documentation: run `dive docs`, or read `docs/index.html` in the repo.
